In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px

## Importing datasets

In [2]:
df_clients = pd.read_csv("../data/raw/clients.csv")
df_products = pd.read_csv('../data/raw/products.csv')
df_stocks = pd.read_csv("../data/raw/stocks.csv")
df_stores = pd.read_csv("../data/raw/stores.csv")
df_transactions = pd.read_csv("../data/raw/transactions.csv")

## Quick checks on duplicates/missing values/etc

In [3]:
df_clients.shape

(424037, 7)

In [4]:
dfs = {
    "clients": df_clients,
    "products": df_products,
    "stocks": df_stocks,
    "stores": df_stores,
    "transactions": df_transactions
}

for name, df in dfs.items():
    print(f"\n=== {name.upper()} DATAFRAME ===")

    duplicates_nb = df.duplicated().sum()
    print(f"The dataframe has {duplicates_nb} duplicated rows among {df.shape[0]} rows")
    if duplicates_nb > 0 :

        df.drop_duplicates(inplace=True)
        print(f"New df has now {df.shape[0]} rows after removing duplicates")

    missing_values = df.isna().sum().sum()
    if missing_values > 0:
        print("Missing values per column:")
        print(df.isna().sum())



=== CLIENTS DATAFRAME ===
The dataframe has 0 duplicated rows among 424037 rows
Missing values per column:
ClientID                 0
ClientSegment            0
ClientCountry            0
ClientOptINEmail         0
ClientOptINPhone         0
ClientGender         60795
Age                 304075
dtype: int64

=== PRODUCTS DATAFRAME ===
The dataframe has 0 duplicated rows among 47458 rows

=== STOCKS DATAFRAME ===
The dataframe has 0 duplicated rows among 16024 rows

=== STORES DATAFRAME ===
The dataframe has 0 duplicated rows among 606 rows

=== TRANSACTIONS DATAFRAME ===
The dataframe has 5121 duplicated rows among 1177175 rows
New df has now 1172054 rows after removing duplicates


Too many missing value in the column age of df_clients, let's get rid of this. Regarding Client Gender, for now we will replace the missing values with "Unkown"

In [5]:
df_clients.drop(columns = "Age", inplace=True)

In [6]:
df_clients["ClientGender"].value_counts()

ClientGender
F    239669
M    122210
C       945
N       216
U       202
Name: count, dtype: int64

In [7]:
df_clients["ClientGender"] = df_clients["ClientGender"].fillna("N/A")

## First visuals insights

### Focus on df_clients

In [8]:
import plotly.express as px

top_n = 15
for col in ["ClientSegment", "ClientCountry", "ClientGender"]:
    plot_df = df_clients.groupby(col).agg(customers_count=(col, "count")).sort_values("customers_count", ascending = False).reset_index().head(top_n)

    fig = px.bar(
        plot_df,
        x=col,
        y="customers_count",
        title=f"{col} by number of customers",
        labels={
            "customers_count": "Number of customers"
        }
    )

    fig.update_layout(
        xaxis_tickangle=-45,
        template="plotly_white"
    )

    fig.show()


### Focus on df_products

In [10]:
import plotly.express as px

top_n = 15
for col in ["Category", "FamilyLevel1", "FamilyLevel2", "Universe"]:
    plot_df = df_products.groupby(col).agg(product_count=(col, "count")).sort_values("product_count", ascending = False).reset_index().head(top_n)

    fig = px.bar(
        plot_df,
        x=col,
        y="product_count",
        title=f"{col} by number of products concerned",
        labels={
            "product_count": "Number of products concerned"
        }
    )

    fig.update_layout(
        xaxis_tickangle=-45,
        template="plotly_white"
    )

    fig.show()


### Focus on df_stocks

In [11]:
df_stocks.head()

,StoreCountry,ProductID,Quantity
0,AUS,1284651161701379667,2.0
1,AUS,6076274819885027797,2.0
2,AUS,6019386668821120661,2.0
3,AUS,2122575437123245322,2.0
4,AUS,5901681811213086415,2.0


In [12]:
df_products.head()

,ProductID,Category,FamilyLevel1,FamilyLevel2,Universe
0,43220326960179274,Football,Ball,Nike Ordem V,Women
1,622915065731236396,Football,Ball,Nike Ordem V,Men
2,2020543468978812774,Football,Shorts,Nike Dri-FIT,Women
3,600002891277549143,Football,Shorts,Nike Dri-FIT,Women
4,6150916997899913693,Football,Shorts,Nike Dri-FIT,Men


In [15]:
df_stocks_merged = df_stocks.merge(df_products, on="ProductID", how="left")

In [16]:
import plotly.express as px

top_n = 15
for col in ["StoreCountry","Category", "FamilyLevel1", "FamilyLevel2", "Universe"]:
    plot_df = df_stocks_merged.groupby(col).agg(available_stock=("Quantity", "sum")).sort_values("available_stock", ascending = False).reset_index().head(top_n)

    fig = px.bar(
        plot_df,
        x=col,
        y="available_stock",
        title=f"Available_stock per {col} overall",
    )

    fig.update_layout(
        xaxis_tickangle=-45,
        template="plotly_white"
    )

    fig.show()


#### Focus per country

In [17]:
top_n = 15

for col in ["Category", "FamilyLevel1", "FamilyLevel2", "Universe"]:

    plot_df = (
        df_stocks_merged
        .groupby([col, "StoreCountry"], as_index=False)
        .agg(available_stock=("Quantity", "sum"))
    )

    # keep only top N values of the main dimension
    top_values = (
        plot_df.groupby(col)["available_stock"]
        .sum()
        .sort_values(ascending=False)
        .head(top_n)
        .index
    )

    plot_df = plot_df[plot_df[col].isin(top_values)]

    fig = px.bar(
        plot_df,
        x=col,
        y="available_stock",
        color="StoreCountry",          # 👈 stacking happens here
        title=f"Available stock per {col}, split by country",
    )

    fig.update_layout(
        xaxis_tickangle=-45,
        template="plotly_white",
        barmode="stack"                # 👈 explicit, though default
    )

    fig.show()


In [18]:
df_transactions.head()

,ClientID,ProductID,SaleTransactionDate,StoreID,Quantity,SalesNetAmountEuro
0,8119209481417068505,3532473209579560668,2023-06-06 00:00:00+00:00,4821951108133690356,4,56.97
1,2497726585282787281,5103640511191568912,2023-09-20 00:00:00+00:00,1450109522794525790,1,5.99
2,7673687066317773168,4923931302917549451,2023-12-16 00:00:00+00:00,1821464542701843363,2,16.99
3,1873234305263900608,8502620308847538595,2023-01-31 00:00:00+00:00,2686511472610728845,4,140.97
4,3913817537779196185,8573693021421318503,2024-01-23 00:00:00+00:00,3600233866627167751,1,10.99


### Aggregating df_transactions with the other dfs

In [20]:
tx = (
    df_transactions
    .merge(df_stores[["StoreID", "StoreCountry"]], on="StoreID", how="left")
)
tx = tx.merge(
    df_products[["ProductID", "Category", "FamilyLevel1", "FamilyLevel2", "Universe"]],
    on="ProductID",
    how="left"
)
tx = tx.merge(
    df_clients[["ClientID", "ClientCountry", "ClientSegment", "ClientGender",
                "ClientOptINEmail", "ClientOptINPhone"]],
    on="ClientID",
    how="left",
    suffixes=("", "_client")
)
# Finally, let's clean date informations
tx["SaleTransactionDate"] = pd.to_datetime(tx["SaleTransactionDate"])
tx["month"] = tx["SaleTransactionDate"].dt.to_period("M").dt.to_timestamp()


/var/folders/dr/m9v4shms079d9s6z6c8cg57m0000gn/T/ipykernel_30453/1645504596.py:21: UserWarning:

Converting to PeriodArray/Index representation will drop timezone information.



In [31]:
tx.to_csv("../data/transformed/tx.csv")